In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## Hatua ya 1: Fafanua Mifano ya Pydantic kwa Matokeo Yenye Muundo

Mifano hii inafafanua **mchoro** ambayo mawakala watarudisha. Kutumia `response_format` na Pydantic huhakikisha:
- ✅ Utoaji data uliolindwa kwa aina
- ✅ Uthibitishaji wa moja kwa moja
- ✅ Hakuna makosa ya kuchambua kutoka kwa majibu ya maandishi huru
- ✅ Routing rahisi ya sharti kulingana na sehemu


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Hatua ya 2: Tengeneza Chombo cha Kuhifadhi Hoteli

Kifaa hiki ndicho **mwakala_wa_upatikanaji** atakachokitumia kuangalia kama vyumba vinapatikana. Tunatumia mshoni wa `@ai_function` ili:
- Geuza kazi ya Python kuwa chombo kinachoitwa na AI
- Tengeneza moja kwa moja muundo wa JSON kwa LLM
- Shughulikia uthibitisho wa vigezo
- Ruhusu mwito wa kiotomatiki na maajenti

Kwa maonyesho haya:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → Kuna vyumba ✅
- **Miji mingine yote** → Hakuna vyumba ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Hatua ya 3: Tafsiri Kazi za Hali kwa ajili ya Uelekezaji

Kazi hizi huchunguza majibu ya wakala na kuamua ni njia gani itachukuliwa katika mtiririko wa kazi.

**Mfumo Muhimu:**
1. Kagua kama ujumbe ni `AgentExecutorResponse`
2. Fanya uchambuzi wa matokeo yaliyopangwa (mfano wa Pydantic)
3. Rudisha `Kweli` au `Uongo` kudhibiti uelekezaji

Mtiririko wa kazi utathibitisha masharti haya kwenye **mipaka** kuamua ni mtendaji gani atakayeteuliwa ifuatayo.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Hatua ya 4: Tengeneza Mfanyakazi wa Maonyesho Maalum

Wafanyakazi ni sehemu za mchakato wa kazi zinazofanya mabadiliko au athari za pembeni. Tunatumia vipengele vya `@executor` kuunda mfanyakazi maalum unaoonyesha matokeo ya mwisho.

**Dhana Muhimu:**
- `@executor(id="...")` - Hurejista kazi kama mfanyakazi wa mchakato wa kazi
- `WorkflowContext[Never, str]` - Vidokezo vya aina kwa ingizo/tokeo
- `ctx.yield_output(...)` - Hutupa matokeo ya mwisho ya mchakato wa kazi


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Hatua ya 5: Pakia Vigezo vya Mazingira

Sanidi mteja wa LLM. Mfano huu unafanya kazi na:
- **Microsoft Foundry** / **Azure OpenAI** (API ya Majibu — inayopendekezwa)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Hatua ya 6: Unda Wakala wa AI wenye Matokeo Yaliyopangwa

Tunaunda **wakala maalum watatu**, kila mmoja akiwa umefungwa ndani ya `AgentExecutor`:

1. **availability_agent** - Huchunguza upatikanaji wa hoteli kwa kutumia zana
2. **alternative_agent** - Hupendekeza miji mbadala (wakati hakuna vyumba)
3. **booking_agent** - Huhamasisha uhifadhi (wakati vyumba vinapatikana)

**Vipengele Muhimu:**
- `tools=[hotel_booking]` - Hutoa zana kwa wakala
- `response_format=PydanticModel` - Inalazimisha matokeo ya JSON yaliyopangwa
- `AgentExecutor(..., id="...")` - Hufunga wakala kwa ajili ya matumizi ya mtiririko wa kazi


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Hatua ya 7: Jenga Mtiririko wa Kazi na Mipaka ya Masharti

Sasa tunatumia `WorkflowBuilder` kujenga grafu yenye njia za masharti:

**Muundo wa Mtiririko wa Kazi:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Njia Muhimu:**
- `.set_start_executor(...)` - Inaweka sehemu ya kuingilia
- `.add_edge(from, to, condition=...)` - Inaongeza mpaka wa masharti
- `.build()` - Hitimisha mtiririko wa kazi


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Hatua ya 8: Endesha Kesi ya Mtihani 1 - Jiji BILA Upatikanaji (Paris)

Hebu tujaribu njia ya **hakuna upatikanaji** kwa kuomba hoteli huko Paris (ambayo haina vyumba katika uigaji wetu).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Hatua ya 9: Endesha Kesi ya Mtihani 2 - Jiji LILIYONAO Upatikanaji (Stockholm)

Sasa wacha tujaze njia ya **upatikanaji** kwa kuomba hoteli huko Stockholm (ambayo ina vyumba katika mfano wetu).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Mambo Muhimu na Hatua Zinazofuata

### ✅ Umejifunza Nini:

1. **Mfumo wa WorkflowBuilder**
   - Tumia `.set_start_executor()` kuweka sehemu ya kuanzisha
   - Tumia `.add_edge(from, to, condition=...)` kwa njia ya masharti
   - Piga simu `.build()` kumaliza mchakato wa workflow

2. **Upangaji wa Masharti**
   - Kazi za sharti huchunguza `AgentExecutorResponse`
   - Tafsiri matokeo yaliyo na muundo kufanya maamuzi ya njia
   - Rudisha `True` kuwezesha njia, `False` kuruka

3. **Uunganishaji wa Zana**
   - Tumia `@ai_function` kubadilisha kazi za Python kuwa zana za AI
   - Wakaazi hupiga simu zana moja kwa moja wanapohitaji
   - Zana hurudisha JSON ambayo wakaazi wanaweza kutafsiri

4. **Matokeo yaliyo na muundo**
   - Tumia modeli za Pydantic kwa uchimbaji data salama wa aina
   - Weka `response_format=MyModel` wakati wa kuunda wakaazi
   - Tumia `Model.model_validate_json()` kuchambua majibu

5. **Watendaji wa Kipekee**
   - Tumia `@executor(id="...")` kuunda vipengele vya workflow
   - Watendaji wanaweza kubadilisha data au kufanya madhara ya upande
   - Tumia `ctx.yield_output()` kutoa matokeo ya workflow

### 🚀 Matumizi Halisi ya Dunia:

- **Ufikiaji wa Safari**: Angalia upatikanaji, pendekeza mbadala, linganisha chaguzi
- **Huduma kwa Wateja**: Panga kulingana na aina ya tatizo, hisia, kipaumbele
- **Biashara Mtandaoni**: Angalia hesabu, pendekeza mbadala, process oda
- **Udhibiti wa Yaliyomo**: Panga kulingana na alama za uovu, bendera za mtumiaji
- **Mchakato wa Idhini**: Panga kulingana na kiasi, nafasi ya mtumiaji, kiwango cha hatari
- **Usindikaji wa Hatua Nyingi**: Panga kulingana na ubora wa data, ukamilifu

### 📚 Hatua Zinazofuata:

- Ongeza masharti tata zaidi (vigezo vingi)
- Tekeleza mizunguko kwa usimamizi wa hali ya workflow
- Ongeza sub-workflows kwa vipengele vinavyoweza kutumika tena
- Unganisha na API halisi (uhifadhi wa hoteli, mifumo ya hesabu)
- Ongeza usimamizi wa makosa na njia za kuondoka
- Onyesha michakato ya workflow kwa zana za kuonyesha zilizojengwa


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
